# Notebook 2 — IndicF5

- Repo: https://github.com/AI4Bharat/IndicF5
- Model: https://huggingface.co/ai4bharat/IndicF5  (gated — accept terms + `huggingface-cli login`)
- Reference-audio prompted (F5-TTS family). Output 24 kHz mono.
- Trained on Indic only, no Hinglish — pure Roman / English-NE inputs are out-of-distribution by design.


In [ ]:
# === Cell 1: install ===
!pip install -q transformers soundfile
!pip install -q git+https://github.com/AI4Bharat/IndicF5.git

# Login if needed (paste your HF token):
# from huggingface_hub import login; login()


In [ ]:
# Mount Drive (or skip if running locally) and clone the audit folder.
# Adjust this cell to point AUDIT_DIR at wherever audit/ lives in your runtime.
import os
from pathlib import Path

# Two common patterns:
#   1. Colab + Drive: AUDIT_DIR = "/content/drive/MyDrive/hienglish/audit"
#   2. Colab + git clone:
#         !git clone https://github.com/<you>/hienglish.git /content/hienglish
#         AUDIT_DIR = "/content/hienglish/audit"
#   3. Local: AUDIT_DIR = str(Path.cwd().parent / "audit")  (if launched from notebooks/)

AUDIT_DIR = os.environ.get("AUDIT_DIR", "/content/audit")
assert Path(AUDIT_DIR).is_dir(), f"AUDIT_DIR={AUDIT_DIR} missing — set it before running."
print(f"AUDIT_DIR = {AUDIT_DIR}")


In [ ]:
import csv
from pathlib import Path

EVAL_TSV = Path(AUDIT_DIR) / "eval_sentences.tsv"
with open(EVAL_TSV, encoding="utf-8") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

assert len(rows) == 30, f"expected 30 sentences, got {len(rows)}"
print(f"Loaded {len(rows)} sentences from {EVAL_TSV}")
print(rows[0])


In [ ]:
# === Cell 4: load model + reference clip ===
from pathlib import Path
import time, json
import numpy as np
import soundfile as sf
import torch
from transformers import AutoModel

MODEL_NAME = "indicf5"
OUT = Path(AUDIT_DIR) / "results" / MODEL_NAME
OUT.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModel.from_pretrained("ai4bharat/IndicF5", trust_remote_code=True).to(device)

REF_AUDIO = str(Path(AUDIT_DIR) / "reference_audio" / "hindi_ref.wav")
with open(Path(AUDIT_DIR) / "reference_audio" / "hindi_ref.txt", encoding="utf-8") as f:
    REF_TEXT = f.read().strip()
assert REF_TEXT, "reference transcript is empty"
print(f"REF_AUDIO={REF_AUDIO}")
print(f"REF_TEXT={REF_TEXT}")


In [ ]:
# === Cell 5: run inference ===
log = []
for r in rows:
    rid, cat, text = r["id"], r["category"], r["text"]
    t0 = time.time()
    try:
        with torch.inference_mode():
            audio = model(text=text, ref_audio_path=REF_AUDIO, ref_text=REF_TEXT)
        if isinstance(audio, torch.Tensor):
            audio = audio.cpu().numpy()
        if audio.dtype == np.int16:
            audio = audio.astype(np.float32) / 32768.0
        audio = np.asarray(audio, dtype=np.float32).squeeze()
        out_path = OUT / f"{rid}.wav"
        sf.write(out_path, audio, 24000)
        log.append({
            "id": rid, "category": cat,
            "duration_s": float(len(audio) / 24000),
            "elapsed_s": time.time() - t0,
            "status": "ok",
        })
    except Exception as e:
        log.append({"id": rid, "category": cat, "status": "error", "error": str(e)})
        print(f"  [error] {rid}: {e}")


In [ ]:
import json
out_log = Path(OUT) / "log.json"
with open(out_log, "w", encoding="utf-8") as f:
    json.dump(log, f, ensure_ascii=False, indent=2)

n_ok = sum(1 for x in log if x["status"] == "ok")
print(f"{MODEL_NAME}: {n_ok}/30 succeeded — log at {out_log}")
